In [147]:
import os
import pandas as pd
import json
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Load JSON inputs
#params_df = pd.read_json("json_outputs/customer_params_df_clean.json", lines=True)
categories_df = pd.read_json("json_outputs/customer_categories_df_clean.json", lines=True)
regions_df = pd.read_json("json_outputs/customer_regions_df_clean.json", lines=True)
payments_df = pd.read_json("json_outputs/payment_lines_clean.json", lines=True)
rep_df = pd.read_json("json_outputs/representatives_clean.json", lines=True)
customer_df = pd.read_json("json_outputs/customer_df_clean.json", lines=True)[[
    'CUSTOMER_NUMBER', 'CCAT_CODE', 'REGION_CODE', 'REP_CODE',
    'SETTLE_TERMS', 'NORMAL_PAYTERMS', 'DISCOUNT', 'CREDIT_LIMIT'
]]

In [148]:
csv_folder = os.path.join(os.getcwd(), "csv_outputs")
json_folder = os.path.join(os.getcwd(), "json_outputs")

In [149]:
# Load customer master (fact table)
customer_df = pd.read_json("json_outputs/customer_df_clean.json", lines=True)

# Merge rep info into master
customer_df = customer_df.merge(rep_df, on="REP_CODE", how="left")

In [150]:
# Merge core customer data
merged_df = customer_df \
    .merge(regions_df, on="REGION_CODE", how="left") \
    .merge(categories_df, on="CCAT_CODE", how="left")
    
#.merge(params_df[['CUSTOMER_NUMBER', 'PARAMETER', 'PARAMETER_GROUP']], on="CUSTOMER_NUMBER", how="left") \

In [151]:
merged_df.shape

(2657, 19)

In [152]:
merged_df.columns.tolist() 

['CUSTOMER_NUMBER',
 'CCAT_CODE',
 'REGION_CODE',
 'REP_CODE',
 'SETTLE_TERMS',
 'NORMAL_PAYTERMS',
 'DISCOUNT',
 'CREDIT_LIMIT',
 'REP_DESC',
 'COMM_METHOD',
 'COMMISSION',
 'REP_DESC_CLEAN',
 'REP_GROUP',
 'PARAMETER',
 'PARAMETER_GROUP',
 'REGION_DESC',
 'PROVINCE',
 'CCAT_DESC',
 'CCAT_GROUP']

In [153]:
merged_df.drop(columns=[
    "PARAMETER_GROUP"
], inplace=True, errors="ignore")

In [154]:
merged_df.shape

(2657, 18)

In [155]:
merged_df.columns.tolist() 

['CUSTOMER_NUMBER',
 'CCAT_CODE',
 'REGION_CODE',
 'REP_CODE',
 'SETTLE_TERMS',
 'NORMAL_PAYTERMS',
 'DISCOUNT',
 'CREDIT_LIMIT',
 'REP_DESC',
 'COMM_METHOD',
 'COMMISSION',
 'REP_DESC_CLEAN',
 'REP_GROUP',
 'PARAMETER',
 'REGION_DESC',
 'PROVINCE',
 'CCAT_DESC',
 'CCAT_GROUP']

In [156]:
# Fill all NaN/null values in object columns with "Unknown"
merged_df.loc[:, merged_df.select_dtypes(include=['object']).columns] = merged_df.select_dtypes(include=['object']).fillna("Unknown")

In [157]:
merged_df

,CUSTOMER_NUMBER,CCAT_CODE,REGION_CODE,REP_CODE,SETTLE_TERMS,NORMAL_PAYTERMS,DISCOUNT,CREDIT_LIMIT,REP_DESC,COMM_METHOD,COMMISSION,REP_DESC_CLEAN,REP_GROUP,PARAMETER,REGION_DESC,PROVINCE,CCAT_DESC,CCAT_GROUP
0,AACJ01,21,25b,ZZZ5,0,90,0,999999,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
1,AACJC1,21,25b,ZZZ5,0,120,0,999999,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
2,AACJC2,5,20a,CONS4,0,120,0,999999,CONSIGNMENTS BM,Gross Profit,0.5,CONSIGNMENT,Channel: Consignment,Unknown,Unknown,Unknown,Consignment,Channel: Consignment
3,AADPRG,6,21a,XX,0,120,0,999999,HOUSE CONSIGNMENTS,Gross Profit,0.0,CONSIGNMENT,Channel: Consignment,Unknown,Unknown,Unknown,Advertising Appro,Internal: Advertising
4,AAMI01,41,10a,02,0,120,0,2000,R,Sales,0.5,R,Sales Rep,Unknown,Durban,KwaZulu-Natal,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2652,ZHAY02,19,4b,03,0,120,0,2000,BJ,Sales,0.5,BJ,Sales Rep,Unknown,Nelspruit / Tzaneen,Mpumalanga,Unknown,Unknown
2653,ZMAU01,37,11a,03,0,120,0,0,BJ,Sales,0.5,BJ,Sales Rep,Unknown,Free State / Lesotho,Free State,Unknown,Unknown
2654,ZNAE01,46,2b,05,0,120,0,30000,RL,Sales,0.5,RL,Sales Rep,Unknown,Krugersdorp / Sun City,North West,Unknown,Unknown
2655,ZNAEOC,5,20a,STAND,0,120,0,999999,STAND-60PC CONSIGNMENT,Gross Profit,0.0,CONSIGNMENT_STANDS,Channel: Consignment,Unknown,Unknown,Unknown,Consignment,Channel: Consignment


In [ ]:
merged_df.to_csv(os.path.join(csv_folder, "customer_merged.csv"), index=False)

# Build JSON records that match the required schema and write newline-delimited JSON
def num_or_none(v):
    if pd.isna(v):
        return None
    try:
        f = float(v)
        return int(f) if f.is_integer() else f
    except Exception:
        return v

records = []
for _, r in merged_df.iterrows():
    records.append({
        "customer_num": None if pd.isna(r.get("CUSTOMER_NUMBER")) else r.get("CUSTOMER_NUMBER"),
        "customer_categories": {
            "ccat_code": num_or_none(r.get("CCAT_CODE")) if "CCAT_CODE" in merged_df.columns else None,
            "ccat_desc": None if pd.isna(r.get("CCAT_DESC")) else r.get("CCAT_DESC") if "CCAT_DESC" in merged_df.columns else None,
            "ccat_group": None if pd.isna(r.get("CCAT_GROUP")) else r.get("CCAT_GROUP") if "CCAT_GROUP" in merged_df.columns else None
        },
        "region": {
            "region_code": None if pd.isna(r.get("REGION_CODE")) else r.get("REGION_CODE") if "REGION_CODE" in merged_df.columns else None,
            "region_desc": None if pd.isna(r.get("REGION_DESC")) else r.get("REGION_DESC") if "REGION_DESC" in merged_df.columns else None,
            "province": None if pd.isna(r.get("PROVINCE")) else r.get("PROVINCE") if "PROVINCE" in merged_df.columns else None
        },
        "rep_code": None if pd.isna(r.get("REP_CODE")) else r.get("REP_CODE"),
        "credit_limit": num_or_none(r.get("CREDIT_LIMIT")) if "CREDIT_LIMIT" in merged_df.columns else None,
        "settle_terms": num_or_none(r.get("SETTLE_TERMS")) if "SETTLE_TERMS" in merged_df.columns else None,
        "normal_payterms": num_or_none(r.get("NORMAL_PAYTERMS")) if "NORMAL_PAYTERMS" in merged_df.columns else None,
        "discount": num_or_none(r.get("DISCOUNT")) if "DISCOUNT" in merged_df.columns else None,
        "parameter": None if pd.isna(r.get("PARAMETER")) else r.get("PARAMETER") if "PARAMETER" in merged_df.columns else None
    })

out_path = os.path.join(json_folder, "customer_merged.json")
with open(out_path, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

# Optionally inspect first record
records[:1]